In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import matplotlib.pyplot as plt
import warnings
import psycopg2
warnings.filterwarnings("ignore")



#analytics_mongo = client_db.get_database('analytics_db')

db = client.get_database('telematics_db')

db2 = client2.get_database('telematics_db')

wave=app_mongo.get_database('app_db')



In [ ]:
from google.cloud import vision
#from google.cloud.vision import types
#from PIL import Image, ImageDraw
import urllib.request
from PIL import Image
import re
import pandas as pd

In [ ]:
#ADHAAR CARD
def adhaar_card(url,op_id):
    try:
        client = vision.ImageAnnotatorClient()
        response = client.text_detection(image=urllib.request.urlopen(url))
        a=response.text_annotations[0].description
        try:
            adhaar_number=re.findall(r'(\d\d\d\d \d\d\d\d \d\d\d\d)',a)[0]
        except Exception as e:
            print("adhaar_card->adhaar_number : ",e)
            adhaar_number=None
        try:
            DOB=re.findall(r'(\d\d/\d\d/\d\d\d\d)',a)[0]
        except Exception as e:
            print("adhaar_card->DOB : ",e)
            DOB=None
        try:
            male_female=re.findall(r'(Male|Female|MALE|FEMALE)',a)[0]  
        except Exception as e:
            print("adhaar_card->male_female : ",e)
            male_female=None        
        return  [adhaar_number,DOB,male_female,op_id,'adhaar_card']

    except Exception as e:
        print('error in adhaar card is '+str(e))
        

In [ ]:
def front_pan_card(url,op_id):
    client = vision.ImageAnnotatorClient()
    response = client.text_detection(image=urllib.request.urlopen(url))
    count=0
    for i in response.text_annotations[0].description.splitlines():
        count+=1
        try:
            if len(re.findall(r'count|rmanent|Account|Number|umber',i))>0:
                PAN_NUMBER=response.text_annotations[0].description.splitlines()[count]
        except:
            PAN_NUMBER=None
    try:
        
        DOB=re.findall(r'\d\d/\d\d/\d\d\d\d',i)[0]
    except:
        DOB=None
    return [PAN_NUMBER,DOB,op_id,'pan_card']   


In [ ]:
def back_adhaar(url,op_id):
    client = vision.ImageAnnotatorClient()
    response = client.text_detection(image=urllib.request.urlopen(url))
    a=response.text_annotations[0].description
    try:
        phone_number=re.findall(r'\d\d\d\d\d\d\d\d\d\d',a)[0]
    except:
        phone_number=None
    return [phone_number,op_id,'adhaar_card']
    

In [ ]:
def get_operator_details(op_id):
    global operators_final
    try:
        adhaar1=adhaar_card(operators_final[operators_final['id']==op_id]['front_adhar_img'].iloc[0],op_id)
        adhaar2=back_adhaar(operators_final[operators_final['id']==op_id]['back_adhar_img'].iloc[0],op_id)
        pan_card=front_pan_card(operators_final[operators_final['id']==op_id]['front_pan_img'].iloc[0],op_id)
        final=[adhaar1[0],adhaar1[1],adhaar1[2],adhaar2[0],pan_card[0],pan_card[1],op_id]
        print(final)
    except:
        final=None
    return final


for i in vehicles.front_rc_img.tolist():
    url=i
    client = vision.ImageAnnotatorClient()
    response = client.text_detection(image=urllib.request.urlopen(url))
    a=response.text_annotations[0].description
    print(url)
    print(len(a))


In [ ]:
final=pd.DataFrame()
for i in range (0,len(operators_final[operators_final['front_adhar_img'].notnull()])+10,10):
    import multiprocessing
    pool = multiprocessing.Pool()
    results=pool.map(get_operator_details,operators_final[operators_final['front_adhar_img'].notnull()]['id'][i:i+10])
    pool.close()
    pool.join()
    results=[i for i in results if i!=None]
    data=pd.DataFrame(results,columns=['adhaar_number','DOB1','Gender','phone_number','PAN_number','DOB2','operator_id'])  
    final=pd.concat([final,data])


In [ ]:
len(operators_final[operators_final['front_adhar_img'].notnull()])

data=pd.DataFrame(results,columns=['adhaar_number','DOB1','Gender','phone_number_from_adhaar','PAN_number','DOB2','operator_id'])  



#selecting partner bank fastag+gps operators

vehicles_partner=pd.read_sql("select vehicle_id from utilisation_data where financer_name='PARTNER_BANK' and trailing_quater=1",con=engine)    
operators_partner=pd.read_sql("select operator_id from operator_vehicles where vehicle_id in {} and service_type='GPS'".format(tuple(vehicles_partner['vehicle_id'])),con=mysql_vds)  
operators_partner_gps_fastag=pd.read_sql("select * from operator_vehicle_detail where operator_id in {}".format(tuple(operators_partner['operator_id'])),con=mysql_vds)    
operators_final=pd.read_sql("select * from operators where id in {}".format(tuple(operators_partner_gps_fastag['operator_id'].unique())),con=mysql_vds)  




In [ ]:
operators_final=pd.read_sql(("select * from fleet_lake.operators where id in (select operator_id from analytics.vehicle_details where sales_source='SAMPLE_CHANNEL')"),con) 



In [ ]:
operators_final

In [ ]:
operators_final=operators_final[['id', 'name', 'company_name',
       'phone_number', 'code',
       'front_pan_img', 'back_adhar_img', 'front_adhar_img', 'gender', 'address1', 'address2', 'address3', 'department']]


In [ ]:
final=pd.merge(final,operators_final,left_on='operator_id',right_on='id',how='left')

In [ ]:
final.to_csv("bureau_data_output.csv")